# VoiceForge — GPU backend

Runs Chatterbox Multilingual voice cloning on a free T4 GPU (Colab or Kaggle) and exposes it as a public HTTPS API via a Cloudflare tunnel.

**Before running:** set the GPU accelerator (Colab: Runtime → Change runtime type → T4 GPU. Kaggle: session settings → Accelerator → GPU T4 x2/P100, and Internet → On).

Then **Run all**, top to bottom, in one pass — no restart needed. Cell 3 prints the backend URL — paste it into VoiceForge. Keep this tab open while generating; the URL is new every session.

**If you ever restart the runtime/session for any reason, re-run Cell 1 first.** A restart can wipe installed packages (this happens on Kaggle); skipping straight to Cell 2 afterward fails with `ModuleNotFoundError: No module named 'chatterbox'`.

> Qwen3-TTS is not installed by default: its dependencies conflict with Chatterbox's (they pin incompatible `transformers` versions) and the two cannot share one environment. Chatterbox Multilingual (23 languages, including English and Hindi) is the recommended narration model.

In [ ]:
# Cell 1 — Install
!pip install -q chatterbox-tts fastapi uvicorn nest-asyncio python-multipart soundfile
# chatterbox-tts pins torch==2.6.0, but Colab/Kaggle preinstall a torchvision
# built for a different torch — mismatched builds break model loading.
# torchvision 0.21.0 is the official pair for torch 2.6.0.
!pip install -q torchvision==0.21.0
# The line above (and Kaggle/Colab's huge preinstalled package set) pulls
# numpy/transformers/safetensors to newer versions than chatterbox-tts
# actually declares — pip's resolver silently leaves them mismatched
# instead of erroring. That breaks the first numpy.random import with:
#   ValueError: numpy.dtype size changed... Expected 96... got 88
# and/or breaks chatterbox's own model-loading code against a transformers
# API it wasn't built for. Fix: pin all three to chatterbox-tts's OWN
# declared requirements (see pypi.org/pypi/chatterbox-tts/json) instead of
# "--upgrade" (which grabs the latest, wrong version).
!pip install -q --force-reinstall numpy==1.26.4 transformers==5.2.0 safetensors==0.5.3
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("Installed. Now run Cell 2 — in the SAME session, no restart needed.")
print("(If you ever restart the runtime/session, re-run Cell 1 first —")
print("never skip straight to Cell 2 after a restart.)")

In [ ]:
# Cell 2 — Load model
import torch, gc
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))

MODELS = {}

# Multilingual Chatterbox: 23 languages including English (en) and Hindi (hi).
# Nepali is not supported by the model. English quality is on par with the
# English-only variant, so this is the only model we load.
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
MODELS["chatterbox"] = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print("chatterbox (multilingual) ready")

# Qwen3-TTS is intentionally not loaded — its `transformers` pin conflicts
# with chatterbox-tts and the two cannot coexist in one environment. The app
# detects which models this backend reports and adapts automatically.

In [ ]:
# Cell 3 — API server + public URL
import io, os, uuid, base64, inspect, subprocess, threading, time, re
import numpy as np, soundfile as sf, torch, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

VOICES = {}   # voice_id -> {"path": wav path, "transcript": str}
os.makedirs("/content/voices", exist_ok=True)

def wav_b64(audio, sr):
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0),
        "models": list(MODELS.keys()),
        "voices": list(VOICES.keys()),
    }

@app.post("/clone")
async def clone(audio: UploadFile = File(...), transcript: str = Form("")):
    vid = uuid.uuid4().hex[:12]
    path = f"/content/voices/{vid}.wav"
    data, sr = sf.read(io.BytesIO(await audio.read()))
    if data.ndim > 1:
        data = data.mean(axis=1)              # to mono
    sf.write(path, data, sr)
    VOICES[vid] = {"path": path, "transcript": transcript.strip()}
    return {"voice_id": vid, "duration": len(data) / sr}

@app.post("/generate")
async def generate(payload: dict):
    text  = payload["text"]
    vid   = payload["voice_id"]
    model = payload.get("model", "chatterbox")
    seed  = int(payload.get("seed", 0))
    if vid not in VOICES:
        return JSONResponse({"error": "unknown voice_id"}, status_code=400)
    if model not in MODELS:
        return JSONResponse({"error": f"{model} not loaded"}, status_code=400)

    torch.manual_seed(seed)
    ref = VOICES[vid]
    t0 = time.time()

    if model == "chatterbox":
        kwargs = {
            "language_id": payload.get("language", "en"),
            "audio_prompt_path": ref["path"],
            "exaggeration": float(payload.get("exaggeration", 0.4)),
            "cfg_weight": float(payload.get("cfg", 0.5)),
            "temperature": float(payload.get("temperature", 0.7)),
        }
        # only pass what this chatterbox build's generate() actually accepts
        accepted = inspect.signature(MODELS["chatterbox"].generate).parameters
        kwargs = {k: v for k, v in kwargs.items() if k in accepted}
        wav = MODELS["chatterbox"].generate(text, **kwargs)
        audio = wav.squeeze(0).cpu().numpy()
        sr = MODELS["chatterbox"].sr
    else:
        wavs, out_sr = MODELS["qwen3"].generate_voice_clone(
            text=text,
            language="English",
            ref_audio=ref["path"],
            ref_text=ref["transcript"] or None,
        )
        audio = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
        if torch.is_tensor(audio):
            audio = audio.cpu().numpy()
        audio = np.asarray(audio, dtype=np.float32).squeeze()
        sr = int(out_sr)

    gc.collect(); torch.cuda.empty_cache()
    return {
        "audio_b64": wav_b64(audio, sr),
        "sample_rate": sr,
        "duration": len(audio) / sr,
        "gen_seconds": round(time.time() - t0, 2),
    }

# --- start server + tunnel ---
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error"),
    daemon=True,
).start()
time.sleep(3)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("  BACKEND URL — paste this into VoiceForge:")
        print("  " + m.group(0))
        print("=" * 60 + "\n")
        print("Keep this tab open while generating.")
        break

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab disconnects idle tabs. Run this in the background during long jobs.
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)